In [2]:
import sys
sys.path.append("../src")   # lets us import our own file from ml/src

import pandas as pd
import preprocessing as prep

df = prep.load_cleaned_data()
print(df.shape)
df.head()

(250000, 17)


,transaction_id,timestamp,transaction_type,merchant_category,amount_inr,transaction_status,sender_age_group,receiver_age_group,sender_state,sender_bank,receiver_bank,device_type,network_type,fraud_flag,hour_of_day,day_of_week,is_weekend
0,TXN0000000001,2024-10-08 15:17:28,P2P,Entertainment,868,SUCCESS,26-35,18-25,Delhi,Axis,SBI,Android,4G,0,15,Tuesday,0
1,TXN0000000002,2024-04-11 06:56:00,P2M,Grocery,1011,SUCCESS,26-35,26-35,Uttar Pradesh,ICICI,Axis,iOS,4G,0,6,Thursday,0
2,TXN0000000003,2024-04-02 13:27:18,P2P,Grocery,477,SUCCESS,26-35,36-45,Karnataka,Yes Bank,PNB,Android,4G,0,13,Tuesday,0
3,TXN0000000004,2024-01-07 10:09:17,P2P,Fuel,2784,SUCCESS,26-35,26-35,Delhi,ICICI,PNB,Android,5G,0,10,Sunday,1
4,TXN0000000005,2024-01-23 19:04:23,P2P,Shopping,990,SUCCESS,26-35,18-25,Delhi,Axis,Yes Bank,iOS,WiFi,0,19,Tuesday,0


In [3]:
X, y = prep.split_x_y(df)

print("X shape:", X.shape, " y shape:", y.shape)
print("Columns in X:", X.columns.tolist())

X shape: (250000, 14)  y shape: (250000,)
Columns in X: ['transaction_type', 'merchant_category', 'amount_inr', 'transaction_status', 'sender_age_group', 'receiver_age_group', 'sender_state', 'sender_bank', 'receiver_bank', 'device_type', 'network_type', 'hour_of_day', 'day_of_week', 'is_weekend']


In [4]:
print("Numeric (will be scaled):", prep.NUMERIC_COLUMNS)
print("Categorical (one-hot):", prep.CATEGORICAL_COLUMNS)
print("Binary (unchanged):", prep.BINARY_COLUMNS)
print("Dropped:", prep.DROP_COLUMNS)

Numeric (will be scaled): ['amount_inr', 'hour_of_day']
Categorical (one-hot): ['transaction_type', 'merchant_category', 'transaction_status', 'sender_age_group', 'receiver_age_group', 'sender_state', 'sender_bank', 'receiver_bank', 'device_type', 'network_type', 'day_of_week']
Binary (unchanged): ['is_weekend']
Dropped: ['transaction_id', 'timestamp']


**Decisions**
- `transaction_id` is just a label and `timestamp` is already represented by
  `hour_of_day`, `day_of_week`, `is_weekend`, so both are dropped from X.
- **One-hot encoding** for text columns (they have no natural order, and each has
  only 3-10 values).
- **StandardScaler** for `amount_inr` and `hour_of_day`.
- `is_weekend` is already 0/1, so it is left unchanged.

In [5]:
X_train, X_test, y_train, y_test = prep.split_train_test(X, y)

print("Train:", X_train.shape, " Test:", X_test.shape)
print("Fraud % in train:", round(y_train.mean() * 100, 3))
print("Fraud % in test :", round(y_test.mean() * 100, 3))

Train: (200000, 14)  Test: (50000, 14)
Fraud % in train: 0.192
Fraud % in test : 0.192


In [6]:
pipeline = prep.build_preprocessor()
pipeline.fit(X_train)          # learns only from the TRAIN data

X_train_final = prep.transform_to_dataframe(pipeline, X_train)
X_test_final = prep.transform_to_dataframe(pipeline, X_test)   # only transform, no fit

print(X_train_final.shape, X_test_final.shape)
display(X_train_final.head())

(200000, 69) (50000, 69)


,amount_inr,hour_of_day,transaction_type_Bill Payment,transaction_type_P2M,transaction_type_P2P,transaction_type_Recharge,merchant_category_Education,merchant_category_Entertainment,merchant_category_Food,merchant_category_Fuel,...,network_type_5G,network_type_WiFi,day_of_week_Friday,day_of_week_Monday,day_of_week_Saturday,day_of_week_Sunday,day_of_week_Thursday,day_of_week_Tuesday,day_of_week_Wednesday,is_weekend
107369,0.002249,-0.904175,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
127423,-0.622335,0.637245,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
181287,-0.623956,0.444568,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
215145,1.285457,1.600632,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
171409,-0.000992,0.829923,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0


In [7]:
# Scaled columns should have mean about 0 and std about 1 on the TRAIN data
display(X_train_final[["amount_inr", "hour_of_day"]].describe().round(2))
print("Missing values in X_train:", X_train_final.isnull().sum().sum())
print("Missing values in X_test :", X_test_final.isnull().sum().sum())

,amount_inr,hour_of_day
count,200000.00,200000.00
mean,-0.00,-0.00
std,1.00,1.00
min,-0.70,-2.83
25%,-0.55,-0.71
50%,-0.37,0.06
75%,0.15,0.83
max,22.04,1.60


Missing values in X_train: 0
Missing values in X_test : 0


In [8]:
X_train_final.to_csv("../../data/processed/X_train.csv", index=False)
X_test_final.to_csv("../../data/processed/X_test.csv", index=False)
y_train.to_csv("../../data/processed/y_train.csv", index=False)
y_test.to_csv("../../data/processed/y_test.csv", index=False)

prep.save_pipeline(pipeline)
print("Everything saved.")

Everything saved.


In [9]:
new_rows = df.head(3).drop(columns=["fraud_flag"])
result = prep.preprocess_new_data(new_rows)
print(result.shape)    # should be (3, 69)

(3, 69)


## Summary
- 200,000 training rows and 50,000 test rows, 69 features after encoding.
- Fraud is 0.192% in both sets (stratified split worked).
- The pipeline was fit on training data only, so there is no data leakage.
- **No SMOTE was applied.** Person 2 should apply it to `X_train` / `y_train` only.
- Saved: `X_train.csv`, `X_test.csv`, `y_train.csv`, `y_test.csv`,
  `ml/artifacts/preprocessing_pipeline.pkl`